# Exploração gráfica quantitativa dos dados originais do INCT SiMAI

Valores absolutos de prédios e pessoas afetados por bairro, a partir de `data_inct_original.json`. Um gráfico por bairro, dois por linha.

In [1]:
import json
from pathlib import Path

import pandas as pd

CAMINHO = Path('..', '..') / 'dados' / 'originais' / 'data_inct_original.json'

with open(CAMINHO, encoding='utf-8') as f:
    raw = json.load(f)

df = pd.DataFrame([
    {'nivel_m': float(nivel.removesuffix('m').replace('_', '.')),
     'bairro': bairro, **valores}
    for nivel, bairros in raw.items()
    for bairro, valores in bairros.items()
])

print(df.shape)

(6674, 7)


## Prédios afetados (valores brutos), um gráfico por bairro

In [2]:
import math
from plotly.subplots import make_subplots

BAIRROS = sorted(df['bairro'].unique())
COR = '#1f77b4'

def graficos_brutos_por_bairro(coluna, titulo_y):
    # Um gráfico por bairro, dois por linha, com valores absolutos
    linhas = math.ceil(len(BAIRROS) / 2)
    fig = make_subplots(rows=linhas, cols=2, subplot_titles=BAIRROS,
                        vertical_spacing=min(0.06, 0.12 / (linhas - 1)), horizontal_spacing=0.08)
    for k, bairro in enumerate(BAIRROS):
        linha, col = k // 2 + 1, k % 2 + 1
        s = df[df['bairro'] == bairro].sort_values('nivel_m')
        fig.add_scatter(x=s['nivel_m'], y=s[coluna], mode='lines+markers', marker=dict(size=3),
                        line=dict(color=COR), showlegend=False, row=linha, col=col)
        fig.add_vline(x=5.35, line_dash='dot', row=linha, col=col)
    fig.update_xaxes(title_text='Nível do Guaíba (m)')
    fig.update_yaxes(title_text=titulo_y)
    fig.update_layout(title=titulo_y, height=280 * linhas + 150, margin=dict(t=100, b=40))
    fig.show()

graficos_brutos_por_bairro('affected_buildings', 'Prédios afetados')

## Pessoas afetadas (valores brutos), um gráfico por bairro

In [3]:
graficos_brutos_por_bairro('affected_population', 'Pessoas afetadas')